In [1]:
# =========================================
# IMPORT LIBRARIES
# =========================================

from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.window import Window

import boto3
from botocore.client import Config

In [2]:
# =========================================
# INIT SPARK SESSION
# =========================================

spark = SparkSession.builder \
    .appName("SV3_Altcoins_ETL") \
    .getOrCreate()

print("Spark Started Successfully")

# =========================================
# MINIO S3A CONFIG
# =========================================

hadoop_conf = spark.sparkContext._jsc.hadoopConfiguration()

hadoop_conf.set("fs.s3a.endpoint", "http://minio:9000")
hadoop_conf.set("fs.s3a.access.key", "admin")
hadoop_conf.set("fs.s3a.secret.key", "password123")
hadoop_conf.set("fs.s3a.path.style.access", "true")
hadoop_conf.set("fs.s3a.connection.ssl.enabled", "false")

hadoop_conf.set(
    "fs.s3a.aws.credentials.provider",
    "org.apache.hadoop.fs.s3a.SimpleAWSCredentialsProvider"
)

print("MinIO Configuration Completed")

Spark Started Successfully
MinIO Configuration Completed


In [3]:
# =========================================
# LOAD RAW DATA FROM MINIO
# =========================================

df = spark.read.csv(
    "s3a://crypto-raw-data/altcoins_500d.csv",
    header=True,
    inferSchema=True
)

print("Raw Dataset Loaded")

print("Total Rows:", df.count())

df.show(20, False)

df.printSchema()

Raw Dataset Loaded
Total Rows: 7000
+-------+----------+------+------+------+------+--------------+
|symbol |timestamp |open  |high  |low   |close |volume        |
+-------+----------+------+------+------+------+--------------+
|ETH/USD|2025-01-27|3228.0|3251.5|3024.0|3181.9|16821.74301423|
|ETH/USD|2025-01-28|3181.7|3224.9|3039.0|3075.8|4970.19146861 |
|ETH/USD|2025-01-29|3077.6|3181.1|3055.0|3114.2|4377.18334445 |
|ETH/USD|2025-01-30|3114.3|3283.1|3092.2|3247.8|5651.86284548 |
|ETH/USD|2025-01-31|3247.2|3437.9|3214.0|3300.1|8634.67643534 |
|ETH/USD|2025-02-01|3299.5|3331.5|3101.6|3116.8|3380.51704713 |
|ETH/USD|2025-02-02|3115.0|3162.5|2751.0|2869.1|14960.17996076|
|ETH/USD|2025-02-03|2869.6|2923.0|2118.0|2883.2|46609.82125989|
|ETH/USD|2025-02-04|2883.1|2891.4|2633.9|2731.4|28393.92306441|
|ETH/USD|2025-02-05|2731.5|2827.5|2700.1|2788.7|18123.51419583|
|ETH/USD|2025-02-06|2788.6|2857.0|2655.3|2687.0|18291.88811049|
|ETH/USD|2025-02-07|2687.4|2798.5|2564.2|2623.5|11100.25994478|
|ETH

In [4]:
# =========================================
# NULL CHECK
# =========================================

print("NULL CHECK")

df.select([
    F.count(
        F.when(F.col(c).isNull(), c)
    ).alias(c)
    for c in df.columns
]).show()

# =========================================
# DUPLICATE CHECK
# =========================================

total_rows = df.count()

unique_rows = df.dropDuplicates(
    ["symbol", "timestamp"]
).count()

print("Total Rows:", total_rows)
print("Unique Rows:", unique_rows)
print("Duplicates:", total_rows - unique_rows)

NULL CHECK
+------+---------+----+----+---+-----+------+
|symbol|timestamp|open|high|low|close|volume|
+------+---------+----+----+---+-----+------+
|     0|        0|   0|   0|  0|    0|     0|
+------+---------+----+----+---+-----+------+

Total Rows: 7000
Unique Rows: 7000
Duplicates: 0


In [5]:
# =========================================
# STANDARDIZE TIMESTAMP
# =========================================

df = df.withColumn(
    "timestamp",
    F.to_timestamp("timestamp")
)

df = df.orderBy(
    "symbol",
    "timestamp"
)

df.select(
    F.min("timestamp").alias("min_time"),
    F.max("timestamp").alias("max_time")
).show(truncate=False)

+-------------------+-------------------+
|min_time           |max_time           |
+-------------------+-------------------+
|2025-01-27 00:00:00|2026-06-10 00:00:00|
+-------------------+-------------------+



In [6]:
# =========================================
# GAP DETECTION
# =========================================

w = Window.partitionBy(
    "symbol"
).orderBy(
    "timestamp"
)

df = df.withColumn(
    "prev_time",
    F.lag("timestamp").over(w)
)

df = df.withColumn(
    "diff_day",
    F.datediff(
        F.col("timestamp"),
        F.col("prev_time")
    )
)

df.select(
    "symbol",
    "timestamp",
    "prev_time",
    "diff_day"
).show(20, False)

gap_count = df.filter(
    F.col("diff_day") > 1
).count()

print("Gap Count:", gap_count)

+--------+-------------------+-------------------+--------+
|symbol  |timestamp          |prev_time          |diff_day|
+--------+-------------------+-------------------+--------+
|AAVE/USD|2025-01-27 00:00:00|NULL               |NULL    |
|AAVE/USD|2025-01-28 00:00:00|2025-01-27 00:00:00|1       |
|AAVE/USD|2025-01-29 00:00:00|2025-01-28 00:00:00|1       |
|AAVE/USD|2025-01-30 00:00:00|2025-01-29 00:00:00|1       |
|AAVE/USD|2025-01-31 00:00:00|2025-01-30 00:00:00|1       |
|AAVE/USD|2025-02-01 00:00:00|2025-01-31 00:00:00|1       |
|AAVE/USD|2025-02-02 00:00:00|2025-02-01 00:00:00|1       |
|AAVE/USD|2025-02-03 00:00:00|2025-02-02 00:00:00|1       |
|AAVE/USD|2025-02-04 00:00:00|2025-02-03 00:00:00|1       |
|AAVE/USD|2025-02-05 00:00:00|2025-02-04 00:00:00|1       |
|AAVE/USD|2025-02-06 00:00:00|2025-02-05 00:00:00|1       |
|AAVE/USD|2025-02-07 00:00:00|2025-02-06 00:00:00|1       |
|AAVE/USD|2025-02-08 00:00:00|2025-02-07 00:00:00|1       |
|AAVE/USD|2025-02-09 00:00:00|2025-02-08

In [7]:
# =========================================
# MA10 & MA60
# =========================================

w10 = Window.partitionBy(
    "symbol"
).orderBy(
    "timestamp"
).rowsBetween(-9, 0)

w60 = Window.partitionBy(
    "symbol"
).orderBy(
    "timestamp"
).rowsBetween(-59, 0)

df = df.withColumn(
    "MA10",
    F.avg("close").over(w10)
)

df = df.withColumn(
    "MA60",
    F.avg("close").over(w60)
)

df.select(
    "symbol",
    "timestamp",
    "close",
    "MA10",
    "MA60"
).show(20, False)

+--------+-------------------+------+------------------+------------------+
|symbol  |timestamp          |close |MA10              |MA60              |
+--------+-------------------+------+------------------+------------------+
|AAVE/USD|2025-01-27 00:00:00|301.8 |301.8             |301.8             |
|AAVE/USD|2025-01-28 00:00:00|281.91|291.855           |291.855           |
|AAVE/USD|2025-01-29 00:00:00|291.21|291.64000000000004|291.64000000000004|
|AAVE/USD|2025-01-30 00:00:00|315.76|297.67            |297.67            |
|AAVE/USD|2025-01-31 00:00:00|332.55|304.646           |304.646           |
|AAVE/USD|2025-02-01 00:00:00|298.25|303.58            |303.58            |
|AAVE/USD|2025-02-02 00:00:00|258.55|297.1471428571429 |297.1471428571429 |
|AAVE/USD|2025-02-03 00:00:00|276.35|294.5475          |294.5475          |
|AAVE/USD|2025-02-04 00:00:00|272.14|292.0577777777778 |292.0577777777778 |
|AAVE/USD|2025-02-05 00:00:00|259.23|288.775           |288.775           |
|AAVE/USD|20

In [8]:
# =========================================
# ROC + MOMENTUM
# =========================================

w = Window.partitionBy(
    "symbol"
).orderBy(
    "timestamp"
)

df = df.withColumn(
    "close_lag10",
    F.lag("close", 10).over(w)
)

df = df.withColumn(
    "ROC",
    (F.col("close") - F.col("close_lag10"))
    / F.col("close_lag10") * 100
)

df = df.withColumn(
    "MOM",
    F.col("close") - F.col("close_lag10")
)

df.select(
    "symbol",
    "close",
    "close_lag10",
    "ROC",
    "MOM"
).show(20, False)

+--------+------+-----------+-------------------+-------------------+
|symbol  |close |close_lag10|ROC                |MOM                |
+--------+------+-----------+-------------------+-------------------+
|AAVE/USD|301.8 |NULL       |NULL               |NULL               |
|AAVE/USD|281.91|NULL       |NULL               |NULL               |
|AAVE/USD|291.21|NULL       |NULL               |NULL               |
|AAVE/USD|315.76|NULL       |NULL               |NULL               |
|AAVE/USD|332.55|NULL       |NULL               |NULL               |
|AAVE/USD|298.25|NULL       |NULL               |NULL               |
|AAVE/USD|258.55|NULL       |NULL               |NULL               |
|AAVE/USD|276.35|NULL       |NULL               |NULL               |
|AAVE/USD|272.14|NULL       |NULL               |NULL               |
|AAVE/USD|259.23|NULL       |NULL               |NULL               |
|AAVE/USD|241.78|301.8      |-19.887342611000665|-60.02000000000001 |
|AAVE/USD|238.38|281

In [9]:
# =========================================
# RSI 14
# =========================================

w1 = Window.partitionBy(
    "symbol"
).orderBy(
    "timestamp"
)

w14 = Window.partitionBy(
    "symbol"
).orderBy(
    "timestamp"
).rowsBetween(-13, 0)

df = df.withColumn(
    "change",
    F.col("close") - F.lag("close").over(w1)
)

df = df.withColumn(
    "gain",
    F.when(
        F.col("change") > 0,
        F.col("change")
    ).otherwise(0)
)

df = df.withColumn(
    "loss",
    F.when(
        F.col("change") < 0,
        -F.col("change")
    ).otherwise(0)
)

df = df.withColumn(
    "avg_gain",
    F.avg("gain").over(w14)
)

df = df.withColumn(
    "avg_loss",
    F.avg("loss").over(w14)
)

df = df.withColumn(
    "RS",
    F.when(
        F.col("avg_loss") == 0,
        None
    ).otherwise(
        F.col("avg_gain") /
        F.col("avg_loss")
    )
)

df = df.withColumn(
    "RSI",
    F.when(
        F.col("avg_loss") == 0,
        100
    )
    .when(
        F.col("avg_gain") == 0,
        0
    )
    .otherwise(
        100 - (100 / (1 + F.col("RS")))
    )
)

print("RSI Created")

df.select(
    "symbol",
    "timestamp",
    "close",
    "RSI"
).show(20, False)

RSI Created
+--------+-------------------+------+------------------+
|symbol  |timestamp          |close |RSI               |
+--------+-------------------+------+------------------+
|AAVE/USD|2025-01-27 00:00:00|301.8 |100.0             |
|AAVE/USD|2025-01-28 00:00:00|281.91|0.0               |
|AAVE/USD|2025-01-29 00:00:00|291.21|31.860226104830332|
|AAVE/USD|2025-01-30 00:00:00|315.76|62.988462969854844|
|AAVE/USD|2025-01-31 00:00:00|332.55|71.7992343683539  |
|AAVE/USD|2025-02-01 00:00:00|298.25|48.306782409615565|
|AAVE/USD|2025-02-02 00:00:00|258.55|35.0377084342351  |
|AAVE/USD|2025-02-03 00:00:00|276.35|42.16103000061603 |
|AAVE/USD|2025-02-04 00:00:00|272.14|41.09523237660621 |
|AAVE/USD|2025-02-05 00:00:00|259.23|38.138757314015045|
|AAVE/USD|2025-02-06 00:00:00|241.78|34.75876079228034 |
|AAVE/USD|2025-02-07 00:00:00|238.38|34.16874687968047 |
|AAVE/USD|2025-02-08 00:00:00|239.69|34.59649818957392 |
|AAVE/USD|2025-02-09 00:00:00|239.16|34.505788067675866|
|AAVE/USD|2025-02-1

In [10]:
# =========================================
# STOCHASTIC OSCILLATOR
# =========================================

df = df.withColumn(
    "highest_high",
    F.max("high").over(w14)
)

df = df.withColumn(
    "lowest_low",
    F.min("low").over(w14)
)

df = df.withColumn(
    "stoch_k",
    F.when(
        (
            F.col("highest_high")
            - F.col("lowest_low")
        ) == 0,
        None
    ).otherwise(
        (
            F.col("close")
            - F.col("lowest_low")
        )
        /
        (
            F.col("highest_high")
            - F.col("lowest_low")
        )
        * 100
    )
)

w3 = Window.partitionBy(
    "symbol"
).orderBy(
    "timestamp"
).rowsBetween(-2, 0)

df = df.withColumn(
    "stoch_d",
    F.avg("stoch_k").over(w3)
)

print("Stochastic Created")

df.select(
    "symbol",
    "timestamp",
    "stoch_k",
    "stoch_d"
).show(20, False)

Stochastic Created
+--------+-------------------+------------------+------------------+
|symbol  |timestamp          |stoch_k           |stoch_d           |
+--------+-------------------+------------------+------------------+
|AAVE/USD|2025-01-27 00:00:00|43.21372854914202 |43.21372854914202 |
|AAVE/USD|2025-01-28 00:00:00|2.3583696488080395|22.786049098975028|
|AAVE/USD|2025-01-29 00:00:00|26.198410663932254|23.923502953960767|
|AAVE/USD|2025-01-30 00:00:00|84.94991448815044 |37.835564933630245|
|AAVE/USD|2025-01-31 00:00:00|76.70336209461473 |62.61722908223248 |
|AAVE/USD|2025-02-01 00:00:00|25.676881880392738|62.44338615438596 |
|AAVE/USD|2025-02-02 00:00:00|11.534287123828319|37.9715103662786  |
|AAVE/USD|2025-02-03 00:00:00|53.75209164628654 |30.32108688350253 |
|AAVE/USD|2025-02-04 00:00:00|51.042605225897795|38.77632799867089 |
|AAVE/USD|2025-02-05 00:00:00|42.73394259235424 |49.17621315484619 |
|AAVE/USD|2025-02-06 00:00:00|31.503410992405716|41.75998627021925 |
|AAVE/USD|2025-

In [11]:
# =========================================
# BUY / SELL LABEL
# =========================================

df = df.withColumn(
    "label",
    F.when(
        F.col("MA10") > F.col("MA60"),
        1
    ).otherwise(0)
)

print("Buy/Sell Label Created")

print("Label Distribution")

df.groupBy(
    "symbol",
    "label"
).count().show()

Buy/Sell Label Created
Label Distribution
+--------+-----+-----+
|  symbol|label|count|
+--------+-----+-----+
|AAVE/USD|    0|  366|
|AAVE/USD|    1|  134|
| ADA/USD|    0|  368|
| ADA/USD|    1|  132|
|ALGO/USD|    0|  342|
|ALGO/USD|    1|  158|
|AVAX/USD|    0|  318|
|AVAX/USD|    1|  182|
| BCH/USD|    0|  265|
| BCH/USD|    1|  235|
|DOGE/USD|    0|  327|
|DOGE/USD|    1|  173|
| DOT/USD|    0|  370|
| DOT/USD|    1|  130|
| ETH/USD|    0|  290|
| ETH/USD|    1|  210|
|LINK/USD|    0|  324|
|LINK/USD|    1|  176|
| LTC/USD|    0|  348|
| LTC/USD|    1|  152|
+--------+-----+-----+
only showing top 20 rows


In [12]:
# =========================================
# FEATURE TABLE
# =========================================

final_df = df.select(
    "symbol",
    "timestamp",
    "open",
    "high",
    "low",
    "close",
    "volume",
    "MA10",
    "MA60",
    "ROC",
    "MOM",
    "RSI",
    "stoch_k",
    "stoch_d",
    "label"
)

print(
    "Rows Before DropNA:",
    final_df.count()
)

final_df = final_df.dropna()

print(
    "Rows After DropNA:",
    final_df.count()
)

final_df.show(
    20,
    False
)

Rows Before DropNA: 7000
Rows After DropNA: 6860
+--------+-------------------+------+------+------+------+--------------+------------------+------------------+-------------------+-------------------+------------------+------------------+------------------+-----+
|symbol  |timestamp          |open  |high  |low   |close |volume        |MA10              |MA60              |ROC                |MOM                |RSI               |stoch_k           |stoch_d           |label|
+--------+-------------------+------+------+------+------+--------------+------------------+------------------+-------------------+-------------------+------------------+------------------+------------------+-----+
|AAVE/USD|2025-02-06 00:00:00|259.34|265.98|239.49|241.78|2638.16533009 |282.773           |284.50272727272727|-19.887342611000665|-60.02000000000001 |34.75876079228034 |31.503410992405716|41.75998627021925 |0    |
|AAVE/USD|2025-02-07 00:00:00|242.44|261.3 |231.44|238.38|1306.91467634 |278.42            

In [13]:
# =========================================
# CREATE MINIO BUCKET
# =========================================

s3 = boto3.client(
    "s3",
    endpoint_url="http://minio:9000",
    aws_access_key_id="admin",
    aws_secret_access_key="password123",
    config=Config(signature_version="s3v4")
)

bucket_name = "crypto-feature-table"

if bucket_name not in [
    b["Name"]
    for b in s3.list_buckets()["Buckets"]
]:
    s3.create_bucket(
        Bucket=bucket_name
    )
    print("Bucket Created")
else:
    print("Bucket Already Exists")

Bucket Already Exists


In [14]:
# =========================================
# SAVE FEATURE TABLE
# =========================================

final_df.write \
    .mode("overwrite") \
    .parquet(
        "s3a://crypto-feature-table/altcoin_features/"
    )

print("Feature Table Saved")

Feature Table Saved


In [15]:
# =========================================
# VERIFY OUTPUT
# =========================================

verify_df = spark.read.parquet(
    "s3a://crypto-feature-table/altcoin_features/"
)

print(
    "Rows Written:",
    verify_df.count()
)

verify_df.show(
    30,
    False
)

verify_df.printSchema()

Rows Written: 6860
+--------+-------------------+------+------+------+------+--------------+------------------+------------------+-------------------+-------------------+------------------+------------------+------------------+-----+
|symbol  |timestamp          |open  |high  |low   |close |volume        |MA10              |MA60              |ROC                |MOM                |RSI               |stoch_k           |stoch_d           |label|
+--------+-------------------+------+------+------+------+--------------+------------------+------------------+-------------------+-------------------+------------------+------------------+------------------+-----+
|AAVE/USD|2025-02-06 00:00:00|259.34|265.98|239.49|241.78|2638.16533009 |282.773           |284.50272727272727|-19.887342611000665|-60.02000000000001 |34.75876079228034 |31.503410992405716|41.75998627021925 |0    |
|AAVE/USD|2025-02-07 00:00:00|242.44|261.3 |231.44|238.38|1306.91467634 |278.42            |280.6591666666667 |-15.441098

In [16]:
# df = spark.read.parquet(
#     "s3a://crypto-feature-table/altcoin_features/"
# )

In [17]:
# df.printSchema()

In [18]:
# df.show(20, truncate=False)